# Fit TabPFN: robot position from lidar

Loads the Unitree A1 datasets generated by `a1_dataset.py` (`datasets/env_A.npz`, `datasets/env_B.npz`),
puts **all** their information into one table (one row per step), and sets up TabPFN to predict the
robot's position (x, y) from the 72 lidar readings.

Table layout: rows are steps. Rows 1-20 are episode 1 of env_A, rows 21-40 episode 2, ...,
then the same for env_B.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "./datasets"
ENVS = ["env_A", "env_B"]
TEST_FRACTION = 0.2   # fraction of episodes (per environment) held out for testing
SEED = 0

## 1. Build the table

Each `.npz` stores observations with shape `(episodes, 21, ...)`: index `t` is the observation *before*
step `t`, index `t+1` the one *after* it. A row for step `t` holds:

| columns | meaning |
|---|---|
| `env`, `episode`, `step` | identifiers (episode and step start at 1) |
| `action_forward`, `action_turn` | command applied during the step |
| `collision`, `collision_box`, `fallen` | robot touched a box during the step / which box (-1 none) / robot fell |
| `lidar_0` ... `lidar_71` | lidar ranges (m) **after** the step, ray `i` at angle `ray_angles[i]` (3.0 = no hit) |
| `x`, `y`, `z`, `qw`, `qx`, `qy`, `qz`, `yaw` | robot pose **after** the step |
| `x_before`, `y_before`, `yaw_before` | robot pose at the start of the step |

Environment-level information (box layout, ray angles, step duration) is not per step and is kept in `env_info`.

In [2]:
def load_env(name: str) -> tuple[pd.DataFrame, dict]:
    d = np.load(f"{DATA_DIR}/{name}.npz")
    E, T = d["collision"].shape
    n_rays = d["lidar"].shape[-1]

    def steps(a):  # (E, T, ...) -> (E*T, ...), episode-major: episode 1 steps 1..T first
        return a.reshape(E * T, *a.shape[2:])

    after = lambda key: steps(d[key][:, 1:])    # observation after each step
    before = lambda key: steps(d[key][:, :-1])  # observation before each step

    table = pd.DataFrame({
        "env": name,
        "episode": np.repeat(np.arange(1, E + 1), T),
        "step": np.tile(np.arange(1, T + 1), E),
        "action_forward": steps(d["action"])[:, 0],
        "action_turn": steps(d["action"])[:, 1],
        "collision": steps(d["collision"]),
        "collision_box": steps(d["collision_box"]),
        "fallen": steps(d["fallen"]),
    })
    lidar = pd.DataFrame(after("lidar"), columns=[f"lidar_{i}" for i in range(n_rays)])
    pose = pd.DataFrame(np.column_stack([after("position"), after("quaternion"), after("yaw")]),
                        columns=["x", "y", "z", "qw", "qx", "qy", "qz", "yaw"])
    pose_before = pd.DataFrame({"x_before": before("position")[:, 0],
                                "y_before": before("position")[:, 1],
                                "yaw_before": before("yaw")})
    info = {"boxes": d["boxes"], "ray_angles": d["ray_angles"], "step_time": float(d["step_time"])}
    return pd.concat([table, lidar, pose, pose_before], axis=1), info


tables, env_info = [], {}
for name in ENVS:
    t, env_info[name] = load_env(name)
    tables.append(t)
df = pd.concat(tables, ignore_index=True)

LIDAR_COLS = [c for c in df.columns if c.startswith("lidar_")]
print(df.shape)  # (2 envs * 300 episodes * 20 steps, columns)
df.head(25)

(12000, 91)


,env,episode,step,action_forward,action_turn,collision,collision_box,fallen,lidar_0,lidar_1,...,y,z,qw,qx,qy,qz,yaw,x_before,y_before,yaw_before
0,env_A,1,1,0.330066,0.663116,False,-1,False,3.000000,3.000000,...,0.595034,0.236591,0.799999,0.004304,-0.028960,-0.599286,-1.286718,2.009890,0.579324,-1.326611
1,env_A,1,2,0.717530,0.406368,False,-1,False,3.000000,3.000000,...,0.597544,0.235469,0.828475,0.031257,-0.058628,-0.556071,-1.185938,1.995769,0.595034,-1.286718
2,env_A,1,3,0.937238,0.263416,False,-1,False,3.000000,2.752667,...,0.532688,0.245456,0.891612,-0.012792,0.035284,-0.451243,-0.938436,1.965749,0.597544,-1.185938
3,env_A,1,4,0.432994,0.674071,False,-1,False,2.775261,2.635084,...,0.432593,0.262179,0.914000,0.018859,-0.020442,-0.404761,-0.834341,2.072214,0.532688,-0.938436
4,env_A,1,5,0.499318,0.180982,False,-1,False,3.000000,2.802686,...,0.355030,0.232312,0.898365,0.010464,0.021279,-0.438609,-0.908371,2.140400,0.432593,-0.834341
5,env_A,1,6,0.706726,0.139877,False,-1,False,2.765755,3.000000,...,0.313741,0.239563,0.927889,0.024056,-0.006645,-0.372021,-0.762483,2.172618,0.355030,-0.908371
6,env_A,1,7,0.674674,0.694088,False,-1,False,3.000000,3.000000,...,0.269937,0.236660,0.946133,-0.000542,0.001797,-0.323773,-0.659433,2.219316,0.313741,-0.762483
7,env_A,1,8,0.651362,0.379432,False,-1,False,3.000000,3.000000,...,0.193619,0.240452,0.966307,0.068483,0.002329,-0.248103,-0.500112,2.283154,0.269937,-0.659433
8,env_A,1,9,0.645913,0.696744,False,-1,False,3.000000,3.000000,...,0.154585,0.245663,0.978525,0.017425,0.002170,-0.205381,-0.413581,2.349151,0.193619,-0.500112
9,env_A,1,10,0.985035,-0.149005,False,-1,False,3.000000,3.000000,...,0.162130,0.237711,0.991259,0.005176,-0.016621,-0.130781,-0.262584,2.450261,0.154585,-0.413581


In [10]:
print(df.columns)
env_A = np.load("./datasets/env_A.npz")
env_B = np.load("./datasets/env_B.npz")
print(env_A.files)

Index(['env', 'episode', 'step', 'action_forward', 'action_turn', 'collision',
       'collision_box', 'fallen', 'lidar_0', 'lidar_1', 'lidar_2', 'lidar_3',
       'lidar_4', 'lidar_5', 'lidar_6', 'lidar_7', 'lidar_8', 'lidar_9',
       'lidar_10', 'lidar_11', 'lidar_12', 'lidar_13', 'lidar_14', 'lidar_15',
       'lidar_16', 'lidar_17', 'lidar_18', 'lidar_19', 'lidar_20', 'lidar_21',
       'lidar_22', 'lidar_23', 'lidar_24', 'lidar_25', 'lidar_26', 'lidar_27',
       'lidar_28', 'lidar_29', 'lidar_30', 'lidar_31', 'lidar_32', 'lidar_33',
       'lidar_34', 'lidar_35', 'lidar_36', 'lidar_37', 'lidar_38', 'lidar_39',
       'lidar_40', 'lidar_41', 'lidar_42', 'lidar_43', 'lidar_44', 'lidar_45',
       'lidar_46', 'lidar_47', 'lidar_48', 'lidar_49', 'lidar_50', 'lidar_51',
       'lidar_52', 'lidar_53', 'lidar_54', 'lidar_55', 'lidar_56', 'lidar_57',
       'lidar_58', 'lidar_59', 'lidar_60', 'lidar_61', 'lidar_62', 'lidar_63',
       'lidar_64', 'lidar_65', 'lidar_66', 'lidar_67', 'lid

In [ ]:
# Sanity checks: 20 consecutive rows per episode, and a quick per-environment summary.
assert (df.groupby(["env", "episode"]).size() == 20).all()
assert (df.loc[:19, "episode"] == 1).all() and (df.loc[20:39, "episode"] == 2).all()

df.groupby("env").agg(
    episodes=("episode", "nunique"),
    rows=("step", "size"),
    collision_rate=("collision", "mean"),
    fallen_rate=("fallen", "mean"),
    n_boxes=("env", lambda s: len(env_info[s.iloc[0]]["boxes"])),
)

## 2. Train / test split by episode

Consecutive steps of an episode are nearly identical, so splitting row by row would leak test
information into training. Whole episodes go to either train or test. The split is stored in the
table (`split` column).

In [ ]:
rng = np.random.default_rng(SEED)
df["split"] = "train"
for name in ENVS:
    episodes = df.loc[df.env == name, "episode"].unique()
    test_eps = rng.choice(episodes, size=int(TEST_FRACTION * len(episodes)), replace=False)
    df.loc[(df.env == name) & df.episode.isin(test_eps), "split"] = "test"

df.groupby(["env", "split"]).size().unstack()

## 3. TabPFN setup

- **Features:** the 72 lidar columns only. **Targets:** `x` and `y` (TabPFN regresses one target at a time,
  so one model per coordinate).
- **One model per environment:** the same scan means different places in different box layouts,
  so the lidar-to-position mapping is environment-specific.
- The lidar is in the robot frame, so the model has to infer the heading implicitly from the scan.
- TabPFN API limit: max 10,000 training rows per fit. Here each fit uses 240 episodes x 20 = 4,800 rows.
  Set `MAX_TRAIN_ROWS` lower to reduce API usage while experimenting.
- Predictions are written back into the table (`x_pred`, `y_pred`, filled for the test rows).

In [ ]:
from tabpfn_client import TabPFNRegressor

TARGETS = ["x", "y"]
MAX_TRAIN_ROWS = 10_000   # TabPFN API limit per fit; lower it to save quota

for target in TARGETS:
    df[f"{target}_pred"] = np.nan

models = {}
for name in ENVS:
    train = df[(df.env == name) & (df.split == "train")]
    test = df[(df.env == name) & (df.split == "test")]
    if len(train) > MAX_TRAIN_ROWS:
        train = train.sample(MAX_TRAIN_ROWS, random_state=SEED)
    for target in TARGETS:
        model = TabPFNRegressor(random_state=SEED)
        model.fit(train[LIDAR_COLS], train[target])
        df.loc[test.index, f"{target}_pred"] = model.predict(test[LIDAR_COLS])
        models[(name, target)] = model
        print(f"{name}: fitted {target} on {len(train)} rows, predicted {len(test)} rows")

## 4. Evaluation

Error metrics on the held-out episodes, compared with a baseline that always predicts the mean
training position.

In [ ]:
test = df[df.split == "test"].copy()
test["error_m"] = np.hypot(test.x_pred - test.x, test.y_pred - test.y)
df.loc[test.index, "error_m"] = test["error_m"]

rows = []
for name in ENVS:
    t = test[test.env == name]
    tr = df[(df.env == name) & (df.split == "train")]
    baseline = np.hypot(tr.x.mean() - t.x, tr.y.mean() - t.y)
    rows.append({
        "env": name,
        "RMSE x (m)": np.sqrt(np.mean((t.x_pred - t.x) ** 2)),
        "RMSE y (m)": np.sqrt(np.mean((t.y_pred - t.y) ** 2)),
        "mean position error (m)": t.error_m.mean(),
        "median position error (m)": t.error_m.median(),
        "baseline mean error (m)": baseline.mean(),
    })
pd.DataFrame(rows).set_index("env").round(3)

In [ ]:
# Predicted vs. true positions on the box layout (requires: pip install matplotlib)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, len(ENVS), figsize=(6 * len(ENVS), 6))
for ax, name in zip(np.atleast_1d(axes), ENVS):
    for bx, by, hx, hy in env_info[name]["boxes"]:
        ax.add_patch(Rectangle((bx - hx, by - hy), 2 * hx, 2 * hy, color="tab:orange"))
    t = test[test.env == name]
    ax.plot([t.x, t.x_pred], [t.y, t.y_pred], color="gray", lw=0.3, alpha=0.5)
    ax.scatter(t.x, t.y, s=4, label="true")
    ax.scatter(t.x_pred, t.y_pred, s=4, label="predicted")
    ax.set_title(f"{name}: test steps (lines join true and predicted)")
    ax.set_aspect("equal")
    ax.legend()
plt.tight_layout()

In [ ]:
# The full table: inputs, everything recorded, split and predictions.
df.head(25)